# Sandbox Notebook: Minimal Solver Layer

Ноутбук использует выделенный модуль `simple_solver_min.py` рядом с файлом.

Пайплайн:
1. Загружаем базовый dataset.
2. Генерируем дневные payload-и.
3. Запускаем solver по дням.
4. Отрисовываем красивую карту за первый день.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import matplotlib.image as mpimg


def find_repo_root(start: Path) -> Path:
    for cand in [start, *start.parents]:
        if (cand / "simple_solver.py").exists() and (cand / "data").exists():
            return cand
    raise RuntimeError("Repo root not found")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
local_pack = Path.cwd().resolve()
repo_pack = REPO_ROOT / "data" / "realistic_spb_sandbox_notebook_data"
if (local_pack / "data" / "dataset_sandbox_type2.json").exists():
    NOTEBOOK_DIR = local_pack
elif (repo_pack / "data" / "dataset_sandbox_type2.json").exists():
    NOTEBOOK_DIR = repo_pack
else:
    raise RuntimeError("Could not locate notebook data directory")

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from simple_solver_min import MultiDaySandboxRunner

DATA_DIR = NOTEBOOK_DIR / "data"
BASE_DATASET_PATH = DATA_DIR / "dataset_sandbox_type2.json"
BASE_DATASET_PATH


## Запуск Multi-day Solving

In [ ]:
OUTPUT_ROOT = NOTEBOOK_DIR / "runtime_outputs"
runner = MultiDaySandboxRunner(BASE_DATASET_PATH, OUTPUT_ROOT)
results = runner.run(days=7, seed=42, mass_noise=0.08, render_first_day=True)

for r in results:
    print(r)


## Метрики По Дням

In [ ]:
days = [r.day_index for r in results]
tr = [r.transport_work_ton_km or 0.0 for r in results]
unassigned = [r.unassigned_tasks for r in results]
checks = [1 if r.all_checks_ok else 0 for r in results]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(days, tr, marker="o", color="#1f77b4")
axes[0].set_title("TR by day (ton*km)")
axes[0].set_xlabel("Day")
axes[0].set_ylabel("TR")
axes[0].grid(alpha=0.2)

axes[1].plot(days, unassigned, marker="s", color="#d62728", label="Unassigned")
axes[1].plot(days, checks, marker="^", color="#2ca02c", label="All checks ok (1/0)")
axes[1].set_title("Daily quality")
axes[1].set_xlabel("Day")
axes[1].grid(alpha=0.2)
axes[1].legend()

plt.tight_layout()
plt.show()


## Красивая Отрисовка За Первый День

In [ ]:
day1_map = OUTPUT_ROOT / "day_001" / "solution_map.png"
print("Day 1 map:", day1_map)

img = mpimg.imread(day1_map)
plt.figure(figsize=(14, 10))
plt.imshow(img)
plt.axis("off")
plt.title("Day 1 route map")
plt.show()
